# Multimodal — โจทย์แบบ "ตอบคำถามจากรูป / อ่านข้อความในรูป" (VQA / OCR)

รูป (+ คำถาม) -> คำตอบสั้น ๆ เช่น "ในรูปมีกี่คน", "ป้ายเขียนว่าอะไร"

วิธีในโน้ตบุ๊กนี้:
- วิธีที่ 1 (ง่ายสุด) = VLM สำเร็จรูปแบบ zero-shot (BLIP-VQA / Qwen2.5-VL) ไม่ต้องเทรน
- วิธีที่ 2 (ขั้นสูง) = fine-tune VLM (ดู reference_cheatsheet.md)


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อมี "รูป + คำถาม" -> ตอบ หรือ "อ่านตัวอักษรในรูป (OCR)"
ถ้าแค่บรรยายรูป (ไม่มีคำถาม) -> `thai_image_captioning.ipynb`

ต้องแก้ (`# << แก้`): ชื่อ competition, path โฟลเดอร์รูป, คอลัมน์คำถาม `Q_COL`, คอลัมน์ id รูป

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install -U transformers pillow pandas torch
!pip -q install pythainlp   # เผื่อวัด/ตัดคำไทย

## ขั้นที่ 2 — ตั้งค่า Kaggle แล้วดาวน์โหลดข้อมูล

ต้องมีไฟล์ `kaggle.json` ก่อน วิธีได้มา: เข้า kaggle.com -> กดรูปโปรไฟล์ -> Settings -> Account -> Create New Token
จะได้ไฟล์ `kaggle.json` หน้าตา `{"username":"...","key":"..."}`

- ถ้ารันบน `Kaggle Notebook`: ข้อมูลอยู่ที่ `/kaggle/input/...` แล้ว ไม่ต้องทำอะไร รันผ่านได้เลย
- ถ้ารันบน `Google Colab / เครื่องตัวเอง`: เอา username กับ key ใส่ในเซลล์ข้างล่าง (จุด `# << แก้`)

In [ ]:
import os, glob, subprocess

COMP = "ใส่-slug-ของ-competition-vqa-ตรงนี้"      # << แก้ตรงนี้: slug ของ competition (คือส่วนท้าย URL หลังคำว่า /competitions/)
KAGGLE_USERNAME = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) เช่น "peetwan1"
KAGGLE_KEY      = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) คีย์ยาว ๆ จาก kaggle.json

def get_data(comp):
    # ถ้าอยู่บน Kaggle ข้อมูลถูกต่อไว้ให้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle แล้ว ข้อมูลอยู่ที่", kpath); return kpath
    # ถ้าไม่ใช่ Kaggle: เขียน token แล้วโหลดด้วย kaggle CLI
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("ไฟล์ที่ดาวน์โหลดมา (ดูชื่อไฟล์/โฟลเดอร์ แล้วเอาไปแก้ในเซลล์ถัดไป):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — path + CONFIG

In [ ]:
import os, glob, pandas as pd, torch
from PIL import Image
TEST_IMG = os.path.join(DATA_DIR,"test")            # << แก้ path รูป
SAMPLE_SUB = os.path.join(DATA_DIR,"sample_submission.csv")
test = pd.read_csv(os.path.join(DATA_DIR,"test.csv"))   # << แก้: ไฟล์ที่มี id รูป + คำถาม
sample = pd.read_csv(SAMPLE_SUB)
IMG_ID_COL = "image"     # << แก้: คอลัมน์ชื่อไฟล์รูปใน test
Q_COL      = "question"  # << แก้: คอลัมน์คำถาม (ถ้าเป็น OCR ล้วน ไม่มีคำถาม ตั้งเป็น None)
ANS_COL    = sample.columns[1]
print("test คอลัมน์:",list(test.columns)); display(test.head()); display(sample.head())

## วิธีที่ 1 — BLIP-VQA สำเร็จรูป (ง่ายสุด ไม่ต้องเทรน)

ตอบคำถามจากรูปแบบ zero-shot สำหรับภาษาไทยที่ซับซ้อนแนะนำเปลี่ยนเป็น Qwen2.5-VL (ดูคอมเมนต์)

In [ ]:
from transformers import pipeline
dev = 0 if torch.cuda.is_available() else -1
# BLIP-VQA (อังกฤษเป็นหลัก). ภาษาไทย/OCR แนะนำใช้ Qwen2.5-VL (โค้ดด้านล่าง)
vqa = pipeline("visual-question-answering", model="Salesforce/blip-vqa-base", device=dev)
ans=[]
for _,r in test.iterrows():
    img = Image.open(os.path.join(TEST_IMG, str(r[IMG_ID_COL]))).convert("RGB")
    q = str(r[Q_COL]) if Q_COL else "What is in the image?"
    ans.append(vqa(image=img, question=q, top_k=1)[0]["answer"])
out=sample.copy(); out[ANS_COL]=ans
out.to_csv("submission.csv",index=False,encoding="utf-8-sig")
print("บันทึก submission.csv"); display(out.head())

## ทางเลือก — Qwen2.5-VL (รองรับไทย + OCR ดีกว่า, ต้อง GPU)

ปลดคอมเมนต์ถ้าต้องการคำตอบภาษาไทยหรืออ่านตัวอักษรในรูป

In [ ]:
# from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
# mdl = Qwen2_5_VLForConditionalGeneration.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct",
#         torch_dtype="auto", device_map="auto")
# proc = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
# def ask(img_path, question):
#     msgs=[{"role":"user","content":[{"type":"image","image":img_path},{"type":"text","text":question}]}]
#     text=proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
#     inputs=proc(text=[text], images=[Image.open(img_path).convert("RGB")], return_tensors="pt").to(mdl.device)
#     out=mdl.generate(**inputs, max_new_tokens=64)
#     return proc.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
print("ปลดคอมเมนต์เซลล์นี้ถ้าจะใช้ Qwen2.5-VL (ภาษาไทย/OCR)")

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
FILE = "submission.csv"   # << แก้เป็นชื่อไฟล์ที่คะแนนดีสุด
!kaggle competitions submit -c {COMP} -f {FILE} -m "blip vqa zero-shot"
!kaggle competitions submissions -c {COMP} | head